# Application of Pathfinding Algorithms: Dijkstra vs A\*

Visualizing Optimal Routes on the Washington DC Street Network

Isak Dai  
May 8, 2026

This paper examines how mapping services use different algorithms to determine optimal routes and estimate travel time. In particular, I compare Dijkstra’s Algorithm with the A-star algorithm. Dijkstra’s algorithm is a search algorithm that iteratively explores the graph until reaching the goal node, while A-star uses a heuristic to guide the search to optimize the search time and memory usage. Both are best-first search algorithms, and Dijkstra’s algorithm can be considerd a special case of A\* where the heuristic estimate of remaining cost to the goal node is 0 for all nodes. I also consider how the choice of costs can impact the results as drivers and pedestrians weight the benefits of optimizing for speed, distance, or incline. I use my commute from home to school as a case study to visualize the results of these algorithms. Finally, I consider the extension of contraction hierarchies to the shortest-path problem, which major mapping services use to speed up route computation.

## Introduction

The origin of shortest-path problems on graphs begins in the 1950s with the work of Edsger Dijkstra \[@dijkstra1959note\], who was one of the first to formulate a formal algorithm for finding the shortest path on a graph. Dijkstra originally developed the algorithm as a toy problem to demonstrate the power of a new computing center in Amsterdam, and used it to find the shortest paths on a pared-down Dutch road network. In 1959, he published the algorithm in a paper intended to minimize the amount of copper wire used in computing chips, but his algorithm now forms the theoretical underpinning for most modern mapping services. \[@frana2010interview\]

Dijkstra’s search algorithm is guaranteed to find the shortest path on a graph, but later researchers began to explore ways to speed up the algorithm and avoid the costly exploration of every node in the graph. Peter Hart, Nils Nilsson, and Bertram Raphael \[@hart1968formal\] developed the A\* algorithm, which uses a so-called hueristic to guide the search. In general terms, A\* prioritizes which nodes to explore by minimizing the estimated total cost of the path to the goal node. As a result, it takes less time and memory to find a path to the goal node.

When it comes to mapping services, there are many choices for how to define the cost of a path. The most simple choice is to use the distance of the path by summing the lengths of the edges in the path from source to goal node. However, in the real world, the fastest route is not always the shortest route. Different roads have different speed limits, some may be unpaved or steep, and some may suffer from traffic congestion. Additionally, there are varied modes of transportation that force different considerations when pathfinding. For example, pedestrians are unlikely to prefer walking on a highway or up a steep hill, even if the physical distance of that path is shorter. Bikers or wheelchair users may also want to avoid steep hills, unpaved roads, or stairs.

This paper will explore the theoretical underpinnings of the Dijkstra and A\* algorithms and implement them in Python on the real-world example of my commute from home to school in Washington, DC. I will demonstrate how different cost models can create different optimal paths. Finally, I will consider the extension of contraction hierarchies to the shortest-path problem, which major mapping services use to speed up route computation.

## Data and graph model

We download a subset of the Washington DC street network from OpenStreetMap \[@openstreetmap\]. OSM data contains a trove of information about the road network, including lengths, speeds, and other features, providing a wealth of differnt options for cost models.

This paper compares two edge-cost models:

-   `length` (meters, baseline shortest distance)
-   `travel_time_s` (seconds, speed-based travel time using OSM `maxspeed` plus highway-class defaults)

In [1]:
from pathlib import Path
import pickle

import pandas as pd
from IPython.display import display

from routing_viz import (
    build_blended_paths_figure,
    build_search_animation,
    comparison_table_row,
)

RESULTS_PKL = Path("data") / "routing_results.pkl"
if not RESULTS_PKL.exists():
    raise FileNotFoundError(
        f"Missing {RESULTS_PKL}. Run `python precompute.py` once before rendering."
    )

with RESULTS_PKL.open("rb") as f:
    bundle = pickle.load(f)

results = bundle["results"]
node_xy_ll = bundle["node_xy_ll"]
edge_pairs = bundle["edge_pairs"]
orig_node = bundle["orig_node"]
dest_node = bundle["dest_node"]
graph_stats = bundle["graph_stats"]
coverage = bundle["coverage"]
blended = bundle.get("blended")

pd.DataFrame(
    [
        {"Point": "origin", "lat": bundle["origin_latlon"][0], "lon": bundle["origin_latlon"][1]},
        {"Point": "destination", "lat": bundle["dest_latlon"][0], "lon": bundle["dest_latlon"][1]},
    ]
)

In [2]:
pd.DataFrame(
    [
        {"Metric": "Nodes", "Value": graph_stats["nodes"]},
        {"Metric": "Directed edges (MultiDiGraph keys)", "Value": graph_stats["edges"]},
        {"Metric": "Edges with maxspeed tag", "Value": coverage["edge_has_maxspeed"]},
        {
            "Metric": "Edges with maxspeed tag (%)",
            "Value": round(100 * coverage["edge_has_maxspeed"] / max(coverage["edge_total"], 1), 2),
        },
    ]
)

## Methods

### Dijkstra’s algorithm

Dijkstra’s algorithm starts out by assigning a distance of 0 to the source node $s$ and an infinite distance to all other nodes. Then, it iteratively calculates the cost of traversing the edge from $s$ to each of its neighbors in a process called “relaxing” the edge. The nodes connected to these edges are placed in a priority queue, with the node having the shortest cost at the front of the queue. Next, the node $v$ at the top of the queue is “settled”, which means that the shortest path from $s$ to $v$ has been found and is stored. The algorithm then relaxes the edges from $v$ to its neighbors, adding the nodes connected to these edges to the priority queue. This process continues until the goal node $t$ is reached. Once it reaches the goal node, it backtracks to construct the shortest path to the source node. \[@dijkstra1959note\] One important caveat is that the edge weights must be non-negative. If there were negative edges, the settling process could break if there were to be a negative edge that created a shorter path to a node that had already been settled.

The use of the priority queue makes Dijkstra’s algorithm a “best-first” search algorithm, but one that is guaranteed to find the shortest path. This is because it has the crucial invariant that the distance to any settled node is the shortest path to that node. Once the target node $t$ is settled, it is thus guaranteed to be the shortest path to the source node $s$.

The method for calculating the total cost of the most efficient path is similar in form to a linked list. Each node remembers its predecessor node on the path with a pointer and stores the cost to get to that node. Starting from the target node $t$, it backtracks to the source node $s$ by following the predecessor pointers to construct the shortest path and its total cost.

### A\*

A\* is an extension of Dijkstra’s algorithm that orders the priority queue not by simple edge weight but by using a heuristic $f(n) = g(n) + h(n)$, where $g(n)$ is the distance from the source node to the node $n$ and $h(n)$ is a heuristic estimate of the remaining cost to the goal \[@hart1968formal\]. Depending on the context, this heuristic can be Euclidean distance, Manhattan distance, or any other heuristic that is guaranteed to be less than or equal to the actual remaining cost to the goal. This requirement is called admissibility. Intuitively, it makes sense that the heuristic must be “optimistic” about the remaining cost to the goal, otherwise the algorithm might prioritize nodes that are further away from the goal node and explore a suboptimal path.

A\* is guaranteed to find the shortest path to the goal node since at any point in the algorithm, it must be true that $g(n)$ is the optimal path to the node $n$. Once the goal node $t$ is settled, $g(t)$ is the shortest path to the source node $s$.

From a different perspective, Dijkstra’s algorithm is a special case of A\* where the heuristic is 0 for all nodes. Since there is no estimate of the remaining distance to the goal node, the algorithm continues to order the priority queue by edge weight. Due to the algorithms’ similarity, I was able to implement both algorithms as variants of a “best-first” search algorithm using a heuristic function, with Dijkstra’s algorithm being a special case where the heuristic is hard-coded as a constant 0 for all nodes.

-   For the `length` model, we use Euclidean distance in projected meters.
-   For the time-based models, we use a lower-bound travel-time heuristic: straight-line distance divided by the maximum edge speed in the graph. This is an optimistic estimate of the remaining travel time to the goal node: you’re driving at highway speeds on the streets of Georgetown!

### Additional cost model

-   **Speed model**: `travel_time_s = length_m / speed_mps` using `maxspeed` when available and fallback defaults by `highway` class.

## Results

In [3]:
rows = []
for model_name, model_res in results.items():
    units = "m" if model_name == "distance" else "s"
    rows.append(
        {
            **comparison_table_row(
                "Dijkstra",
                model_res["dijkstra"].cost,
                model_res["dijkstra"].nodes_settled,
                model_res["dijkstra"].pq_pops,
                model_res["dijkstra"].elapsed_s * 1000,
            ),
            "Model": model_name,
            "Cost units": units,
        }
    )
    rows.append(
        {
            **comparison_table_row(
                "A*",
                model_res["astar"].cost,
                model_res["astar"].nodes_settled,
                model_res["astar"].pq_pops,
                model_res["astar"].elapsed_s * 1000,
            ),
            "Model": model_name,
            "Cost units": units,
        }
    )

pd.DataFrame(rows)

Nodes settled counts the nodes for which the shortest path from my house has been found. PQ pops counts heap pop operations, or the number of times the algorithm considers a node in the priority queue. Time is the computational time in milliseconds. As expected, A\* settles far fewer nodes than Dijkstra’s algorithm due to its goal-directed search. This is true using a distance-based and speed-based cost model. A pure distance-based cost model yields much greater computational improvements for a tradeoff between A\* and Dijkstra’s algorithm (4.2 ms for A\* vs. 22.0 ms for Dijkstra). The speed-based cost model narrows this gap (19.4 ms for A\* vs. 21.7 ms for Dijkstra), as A\* is taken down arterial roads that have a lower speed-based cost to their traversal.

It is easiest to understand the differences in search patterns using the actual map of Washington, DC as shown in the interactive maps below.

In [4]:
MAX_FRAMES = 30


def animation_for(algo: str, model: str, title: str):
    res = results[model][algo]
    return build_search_animation(
        res.frames,
        node_xy_ll,
        edge_pairs,
        res.path,
        orig_node,
        dest_node,
        map_title=title,
        max_frames=MAX_FRAMES,
    )

### Interactive map: Dijkstra (distance baseline)

In [5]:
animation_for("dijkstra", "distance", "Dijkstra (distance) on DC OSM drive network")

### Interactive map: Dijkstra (speed-based time)

In [6]:
animation_for("dijkstra", "time", "Dijkstra (travel_time_s) on DC OSM drive network")

### Interactive map: A\* (distance baseline)

In [7]:
animation_for("astar", "distance", "A* (distance) on DC OSM drive network")

Right off the bat, notice how A\* explores the graph differently than Dijkstra. Thanks to the heuristic, A\* prioritizes exploring nodes that have a shorter straight-line distance to the goal node. There are still a few attempts to venture off into Northeast because the actual distance from my house to those nodes are small enough to be worth exploring. \### Interactive map: A\* (speed-based time)

In [8]:
animation_for("astar", "time", "A* (travel_time_s) on DC OSM drive network")

Here, we see that A\* where speed is taken into account explores a wider range of nodes than A\* based solely on distance. The settled nodes travel down major thoroughfares like 14th Street, Connecticut Avenue, and Georgia Avenue that have a lower cost $g(n)$ due to their higher speed limits..

## Interactive cost mix

This static explorer precomputes shortest paths under a sweep of weight combinations between distance and travel time. Each edge metric (`length` and `travel_time_s`) is normalized to `[0, 1]`, then combined as:

`composite_cost = w_distance * length_norm + w_time * time_norm`,

with `w_time = 1 - w_distance`. Use the slider to shift between distance- and travel-time-priority routing.

In [9]:
if blended is None:
    import plotly.graph_objects as go

    fig = go.Figure()
    fig.update_layout(
        title=(
            "Custom cost mix unavailable. Run `python precompute.py` in your project "
            "environment to regenerate data/routing_results.pkl with blended outputs."
        ),
        margin={"l": 20, "r": 20, "t": 80, "b": 20},
        height=220,
    )
    display(fig)
else:
    fig = build_blended_paths_figure(
        node_xy_ll,
        edge_pairs,
        blended["weights"],
        blended["paths"],
        blended["stats"],
        orig_node,
        dest_node,
        map_title="Custom cost mix (Dijkstra)",
    )
    display(fig)

In [10]:
pd.DataFrame([] if blended is None else blended["stats"])

## Extenstion: Contraction Hierarchies

Contraction hierarchies preprocess the graph so that bidirectional search on a much larger search space still yields exact shortest paths on road networks \[@geisberger2008contraction\]. Nodes can be merged together to create a smaller graph that is still guaranteed to contain the shortest path to the goal node while requiring fewer nodes to be relaxed. For instance, if I want to calculate the path between Georgetown University and Johns Hopkins University, it would be imprudent to explore all the side streets between Washington, DC and Baltimore. Instead, I could precalculate a super node representing the cost of all of I-95 between DC and Baltimore. Althought this is technically adding a node to the graph, it also obviates the need to explore all the nodes on I-95 between DC and Baltimore since we can precompute that distance. As a result, we no longer explore all the offramps since we

Mapping services use contraction hierarchies to make route computation over large distances near-instantaneous. It is possible to precompute the shortest path between many pairs of nodes in a graph, and the hierarchical structure of roads (i.e. local roads feed into arterial roads feed into highways) allows for efficient route computation over large distances. Mapping services use contraction hierarchies with a modified version of Dijkstra’s algorithm to find the shortest path between two nodes. They tend to use bidirectional search (i.e. starting from both the source and goal node and meeting in the middle) and only explore “up” in the hierarchy (i.e. not take a diversion off a side street when already on a major road). With these and other properietary optimazations, mapping services can deliver optimal routes in millseconds for thousands of customers at a time.

## Conclusion

This project considers the development of pathfinding algorithms and their application to real-world examples. I present Dijkstra’s Algorithm and A\* as two extensions of a best-first search algorithm, where Dijkstra’s algorithm is a special case of A\* where the heuristic estimate of remaining cost to the goal node is 0 for all nodes. Additionally, I consider the implications of different cost models on the results of the algorithms. In A*, it is evident that a speed-based cost model results in arterial roads getting more priority in explanation since there is a lower speed-based cost to their traversal. In a purely distance-based cost model, A* is about 5x faster than Dijkstra’s algorithm, but this advantage slims when using a speed-based cost model.

In the case of my commute from home to school, both cost models produce near-identical paths. Since the distance as the crow flies is only ~2.5 miles, there are no highways in the way, and DC has a relatively regular grid street layout, it makes sense that there is not much deviation between the two cost models. However, if we were able to take grade into account, it is possible that there would be a very different route that avoided steep hills. Unfortunately, the OSM data did not include data on the incline of the roads, so that possibility remains unexplored in this project.

Finally, I explore the extension of contraction hierarchies to the shortest-path problem and give a brief overview of how mapping services use them to speed up route computation, reverting to Dijkstra’s algorithm but precomputing many well-traveled routes in advance to allow for near-instantaneous route computation for endpoint users.